<a href="https://colab.research.google.com/github/moizr1732/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


## Finding 1

One finding from the FlyRank research paper is that the proposed machine learning model performed better than the baseline methods for predicting content refresh opportunities.

### My Methodology Question

I would ask how the data was split for training and testing. If data from the same client or the same time period appears in both sets, the results could be overly optimistic. Using a grouped or time-aware split would provide a more realistic evaluation.

---

## Finding 2

Another finding is that some features were more important than others in making predictions.

### My Methodology Question

I would ask whether all of the features were available before making the prediction. If any feature contains information from the future or directly relates to the target variable, it could introduce data leakage and make the model appear more accurate than it really is.

These questions are meant to improve confidence in the research by checking that the methodology supports the reported results.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


My Model Under an Honest Split

I re-evaluated my Week 5 model using a grouped validation split based on client_id. This prevents data from the same client appearing in both the training and test sets, making the evaluation more realistic.

Compared with the original evaluation, the grouped split produced lower performance, which is expected because it is a more challenging and honest validation strategy. This gives a better estimate of how the model may perform on unseen clients.

Results (Grouped Split):

Accuracy: 0.5788
Precision: 0.5461
Recall: 0.5788
F1 Score: 0.4942
ROC-AUC: 0.8246

The results suggest that while the model still shows useful predictive ability, its performance should be interpreted as decision-support rather than proof that it will generalize perfectly to all future datasets.

In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Read dataset directly
df = pd.read_csv("content_refresh_anonymized (1) (2).csv")
print(df.shape)

# Target column
target = "trend_direction"

# Columns to remove
drop_cols = [
    target,
    "client_id",
    "url",
    "trend_pct",
    "is_declining"
]

# Only drop columns that actually exist
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Grouped split
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
# For multiclass ROC-AUC, predict_proba needs to return probabilities for all classes
# and multi_class parameter must be specified
y_prob = model.predict_proba(X_test)

# Metrics
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, average='weighted'), 4))
print("Recall   :", round(recall_score(y_test, y_pred, average='weighted'), 4))
print("F1 Score :", round(f1_score(y_test, y_pred, average='weighted'), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob, multi_class='ovr'), 4))

(30000, 44)
Accuracy : 0.5788
Precision: 0.5461
Recall   : 0.5788
F1 Score : 0.4942
ROC-AUC  : 0.8246


| Metric    | Week 5 (Original Split) | Honest Grouped Split |
| --------- | ----------------------: | -------------------: |
| Accuracy  |              **0.6627** |           **0.5788** |
| Precision |              **0.6312** |           **0.5461** |
| Recall    |              **0.6627** |           **0.5788** |
| F1 Score  |              **0.5915** |           **0.4942** |
| ROC-AUC   |              **0.8975** |           **0.8246** |



The grouped split produces lower evaluation metrics than the original random split. This is expected because the grouped split prevents data leakage by ensuring that records from the same patient do not appear in both the training and testing sets. As a result, the evaluation better reflects the model's ability to generalize to unseen patients, making the reported performance more realistic and trustworthy.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I reviewed the features used in my model to check for possible data leakage.

I confirmed that the target variable was not included as an input feature during training. I also reviewed the remaining features to make sure they represented information that would reasonably be available before making a prediction.

Based on this review, I did not find any obvious evidence of target leakage. However, feature selection should always be reviewed carefully whenever new data or additional features are added to the project.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Claim

The model accurately predicts content refresh opportunities.

### Revised Claim

Based on the evaluation performed in this project, the model showed good predictive performance on the available dataset. The results should be considered decision-support rather than proof that the model will perform equally well in every real-world situation. Additional testing on new and unseen data would provide stronger evidence of its generalization ability.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.